# 26 — de-shortcut eval

Roadmap phase 0.4. **Part 1** (zero GPU) regenerates the two exposure/margin tables that
`tables/` holds. **Part 2** (zero GPU here) builds the paired arms the GPU half will infer,
runs the four blocking gates, and writes the sample a human reads *before* any GPU time.

Pre-registration and the decision rule: `PLAN.md`. This notebook produces **no `bucket_mean`**
and no `RESULTS.csv` row — the rung re-reads an existing scored run and prepares a probe.

In [ ]:
# ── config (inline, EXPERIMENT_REPO_STRUCTURE_SPEC: notebooks carry their own config) ────────
#
# papermill-overridable. On the pod the corpus lives at /workspace/orena-data, NOT inside the
# repo — `external_data/` there is empty, so GOLD_GLOB must be passed.
from pathlib import Path

REPO = Path.cwd().parents[1]
HERE = REPO / "experiments" / "26-deshortcut-eval"

GOLD_GLOB = str(REPO / "external_data/orena-data/*/data/frame/test.parquet")

# Part 1 re-reads an ALREADY SCORED run. It is not a new evaluation: no model is loaded.
# Default is the run part 1 was published against, so `tables/` reproduces bit for bit.
SCORED_CSV = str(REPO / "experiments/02-lora-sft/runs/02_lora_sft_v1/eval_best/results.csv")

WRITE_TABLES = True   # part 1 → tables/exposure.csv, tables/strata_rung02.csv
WRITE_ARMS = True     # part 2 → tables/part2_arms.csv, tables/part2_eyeball.csv
EYEBALL_N = 30

In [ ]:
import sys

sys.path.insert(0, str(HERE / "_models"))
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

SCORED_CSV = Path(SCORED_CSV)
print(f"repo   {REPO}")
print(f"gold   {GOLD_GLOB}")
print(f"scored {SCORED_CSV}  exists={SCORED_CSV.exists()}")

In [ ]:
# ── gold: the two val parquets, with the canonical qID and distribution ──────────────────────
#
# 🔴 The parquet's own `ood` column is all-False in BOTH splits (RULES §3) — it is a landmine,
# not a signal. Distribution comes from the qID prefix, and RULES §1 forbids re-deriving a
# metric here, so we import the canonical implementation rather than write a second one.
import glob

from frame.metrics import _distribution as distribution_of

parts = []
for f in sorted(glob.glob(GOLD_GLOB)):
    dataset = Path(f).parents[2].name          # .../<dataset>/data/frame/test.parquet
    parts.append(pd.read_parquet(f).assign(dataset=dataset))
gold = pd.concat(parts, ignore_index=True)
gold["qID"] = gold["dataset"] + "__" + gold["id"].astype(str)
gold["distribution"] = distribution_of(gold, None)

assert len(gold) == 6252, f"expected the full val set, got {len(gold)}"
assert gold["qID"].is_unique, "qID must key the gold"
assert not gold["ood"].any(), "the parquet `ood` column is expected all-False; RULES §3"

print(gold.groupby(["distribution", "answer_format"]).size().unstack(fill_value=0))

## Part 1 — exposure and the per-stratum margin

Regenerates what `tables/` already holds. Both calls are the engine's; nothing is computed
inline. 🔴 The pooled gap is a format-mix artefact and `strata_report` refuses to emit one —
read the table **within format**, never across.

In [ ]:
from shortcut_taxonomy import exposure, strata_report

exposure_df = exposure(gold)
display(exposure_df)

if not SCORED_CSV.exists():
    raise FileNotFoundError(
        f"{SCORED_CSV} is missing. `runs/` is gitignored, so part 1 regenerates only where the"
        " run artifacts live (the pod volume). Point SCORED_CSV at a scored results.csv."
    )

# results.csv already carries `answer_format`; bring only what it lacks, or the merge would
# silently produce answer_format_x / answer_format_y and strata_report would not find its key.
scored = pd.read_csv(SCORED_CSV).merge(
    gold[["qID", "question", "answer", "distribution"]], on="qID", how="left"
)
assert scored["question"].notna().all(), "every scored row must join a gold question"
assert "answer_format" in scored.columns, "the merge collided on answer_format"

strata_df = strata_report(scored)
display(strata_df)

if WRITE_TABLES:
    (HERE / "tables").mkdir(exist_ok=True)
    exposure_df.to_csv(HERE / "tables/exposure.csv", index=False)
    strata_df.to_csv(HERE / "tables/strata_rung02.csv", index=False)
    print("wrote tables/exposure.csv, tables/strata_rung02.csv")

## Part 2 — the paired arms

Same frame, same gold, three phrasings. Scoped to `fo_class`, the only cell part 1 found an
inflation to explain, and the format that owns half the headline.

| arm | what it is |
|---|---|
| `original` | the corpus question, re-inferred in the same process (not the archived answers — ~0.5% drift on a GPU swap) |
| `premise_dropped` | **primary.** the leading cardinality sentence deleted, nothing else touched |
| `set_framed` | re-asked with the corpus's own shortcut-free `fo_class` template |

🔴 Three of SurgCheck's four grounding cues (box, arrow, spatial position) need localization we
do not have. Both arms use the fourth, periphrasis.

🔴 **The primary read is ID.** OOD carries more questions (465 vs 208) but only **10 videos**
against ID's 28, and the effective n is videos (RULES §13). ID is also the half that populates
the leaderboard's `pre_evaluation_score` today. OOD is reported as secondary and an
`INCONCLUSIVE` there is expected, not a finding.

In [ ]:
from deshortcut import ARMS, assert_arms_wellformed, build_arms, eyeball_sample, shortcut_stratum

stratum = shortcut_stratum(gold)
arms = build_arms(gold)

# Blocking. A malformed rewrite yields a wrong answer that is not the model's fault, and it
# would read as "the margin was phrasing" — the very conclusion part 2 exists to test.
assert_arms_wellformed(arms, gold)

print(f"stratum       {len(stratum)} questions, {stratum['question'].nunique()} distinct template(s)")
print(f"power         {stratum.groupby('distribution')['video'].nunique().to_dict()} videos")
print(f"              {stratum['distribution'].value_counts().to_dict()} questions")
print(f"arms          {len(arms)} rows = {len(stratum)} x {len(ARMS)}")
print(f"gold classes  {stratum['answer'].value_counts().to_dict()}")
print("\nGATES PASSED — no arm carries a shortcut under part 1's own detector")

In [ ]:
# ── the human read, before any GPU time (PLAN.md, as rung 09 stage 1 does) ───────────────────
#
# ⚠️ The stratum is ONE template, so the rewrite is a single deterministic transformation and
# the review collapses from 30 rows to 3 sentences. Signed off by legokna 2026-07-30. The
# sample is still written, because the per-class gold varies and the fairness of `set_framed`
# is a per-class question.
eyeball = eyeball_sample(arms, n=EYEBALL_N)

if WRITE_ARMS:
    (HERE / "tables").mkdir(exist_ok=True)
    cols = ["qID", "video", "dataset", "distribution", "arm", "question", "answer"]
    arms[cols].to_csv(HERE / "tables/part2_arms.csv", index=False)
    eyeball.to_csv(HERE / "tables/part2_eyeball.csv", index=False)
    print("wrote tables/part2_arms.csv, tables/part2_eyeball.csv")

for arm in ARMS:
    q = arms.loc[arms["arm"] == arm, "question"].iloc[0]
    print(f"\n[{arm}]\n  {q}")

## What is still owed

1. ✅ **the human read** — signed off 2026-07-30;
2. **the checkpoint**, per the selection rule declared 2026-07-30: the arm owning
   `object_recognition_ID` at epoch 3, where rung 21's `A3_vitlr` displaces `A2_lr` **only if**
   its paired delta against A2 excludes zero;
3. **the inference** — 2,019 questions, one checkpoint, no training. All three arms in one
   process so the pairing carries no cross-GPU drift;
4. the read: `paired_delta_ci` + `equivalence_verdict` at the pre-declared **ε = 0.05**, against
   `premise_dropped`, **primary on ID**. `INCONCLUSIVE` is not a pass.